<a href="https://colab.research.google.com/github/asjath44/api-ethics-assignment/blob/main/ml_leakage_pipeline_asjath44.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# FULL SOLUTION — ML Leakage & Pipeline
# ============================================

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

# ----------------------------
# Task 1: WRONG (Data Leakage)
# ----------------------------

X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # ❌ WRONG: scaling before split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))

print("=" * 40)
print("TASK 1 — WRONG (DATA LEAKAGE)")
print("=" * 40)
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print("❌ Problem: Scaling done before split → leakage\n")

# ----------------------------
# Task 2: CORRECT (Pipeline)
# ----------------------------

X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

pipeline.fit(X_train, y_train)

train_acc = accuracy_score(y_train, pipeline.predict(X_train))
test_acc = accuracy_score(y_test, pipeline.predict(X_test))

cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5)

print("=" * 40)
print("TASK 2 — CORRECT (PIPELINE)")
print("=" * 40)
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"CV Mean:        {cv_scores.mean():.4f}")
print(f"CV Std:         {cv_scores.std():.4f}")
print("✅ No leakage — scaling only on training data\n")

# ----------------------------
# Task 3: Decision Tree
# ----------------------------

print("=" * 40)
print("TASK 3 — DECISION TREE")
print("=" * 40)
print("Depth | Train Acc | Test Acc")

depths = [1, 5, 20]

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, tree.predict(X_train))
    test_acc = accuracy_score(y_test, tree.predict(X_test))

    print(f"{d:5} | {train_acc:.3f}     | {test_acc:.3f}")

print("\nBest depth = 5 (balance).")

TASK 1 — WRONG (DATA LEAKAGE)
Train Accuracy: 0.8700
Test Accuracy:  0.8300
❌ Problem: Scaling done before split → leakage

TASK 2 — CORRECT (PIPELINE)
Train Accuracy: 0.8700
Test Accuracy:  0.8300
CV Mean:        0.8700
CV Std:         0.0179
✅ No leakage — scaling only on training data

TASK 3 — DECISION TREE
Depth | Train Acc | Test Acc
    1 | 0.882     | 0.840
    5 | 0.954     | 0.855
   20 | 1.000     | 0.840

Best depth = 5 (balance).
